In [1]:
import numpy as np
import pandas as pd

In [2]:
actual=np.array(["cat","dog","dog","cat","dog","dog","cat"])
predict=np.array(["cat","dog","cat","dog","cat","dog","dog"])

In [3]:
pos="dog"

In [4]:
table=[[0,0],[0,0]]
for i in range(len(actual)):
    if actual[i]!=pos:
        if predict[i]!=pos:
            table[0][0]+=1
        else:
            table[0][1]+=1
    else:
        if predict[i]!=pos:
            table[1][0]+=1
        else:
            table[1][1]+=1
print(table)

[[1, 2], [2, 2]]


In [5]:
x=sum(np.array(table).reshape(-1))
accuracy=(table[0][0]+table[1][1])/x
error=1-accuracy
precision=table[1][1]/sum(table[1])
recall=table[1][1]/(table[0][0]+table[0][1])
specifcity=table[0][0]/(table[0][0]+table[1][0])
f1_score=2*precision*recall/(precision+recall)
print(accuracy,error,precision,error,precision,recall,f1_score)

0.42857142857142855 0.5714285714285714 0.5 0.5714285714285714 0.5 0.6666666666666666 0.5714285714285715


In [6]:
import pandas as pd
df=pd.read_csv("diabetes.csv")
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [7]:
import numpy as np
import pandas as pd
import seaborn as sns

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

In [9]:
df=pd.read_csv("diabetes.csv")

In [10]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [11]:
# underweigth<18.5
# healthy weight:18.5-24.9
# overweight:25.0-29.9
# obesity>30

In [ ]:
df["BMI_CAT"]=pd.cut(df.BMI,bins=[0,18.5,24.9,29.9,100],labels=["UnderWeight","healthy weight","overweight","obesity"])
# Another method is apply
# def bmi_cat(x):
#     if x<18.5:
#         return "UnderWeight"
#     elif x<24.9:
#         return "healthy weight"
#     elif x<29.9:
#         return "overweight"
#     else:
#         return "obesity"
# df["BMI_CAT"]=df.BMI.apply(bmi_cat)
# df.head()
#Another method is np.where
# df["BMI_CAT"]=np.where(df.BMI<18.5,"UnderWeight",
#                 np.where(df.BMI<24.9,"healthy weight",
#                 np.where(df.BMI<29.9,"overweight","obesity")))
# Another method is lambda function
# df["BMI_CAT"]=df.BMI.apply(lambda x:"UnderWeight" if x<18.5 else "healthy weight" if x<24.9 else "overweight" if x<29.9 else "obesity")

In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   Pregnancies               768 non-null    int64   
 1   Glucose                   768 non-null    int64   
 2   BloodPressure             768 non-null    int64   
 3   SkinThickness             768 non-null    int64   
 4   Insulin                   768 non-null    int64   
 5   BMI                       768 non-null    float64 
 6   DiabetesPedigreeFunction  768 non-null    float64 
 7   Age                       768 non-null    int64   
 8   Outcome                   768 non-null    int64   
 9   BMI_CAT                   757 non-null    category
dtypes: category(1), float64(2), int64(7)
memory usage: 55.1 KB


In [26]:
df.dropna(inplace=True)


In [29]:
from sklearn.metrics import confusion_matrix


x=df.drop(["BMI_CAT","Outcome"],axis=1)
y=df.Outcome
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
knn=KNeighborsClassifier(n_neighbors=3)
knn.fit(x_train,y_train)
y_pred=knn.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))


[[73 24]
 [26 29]]
0.6710526315789473
              precision    recall  f1-score   support

           0       0.74      0.75      0.74        97
           1       0.55      0.53      0.54        55

    accuracy                           0.67       152
   macro avg       0.64      0.64      0.64       152
weighted avg       0.67      0.67      0.67       152



In [33]:
# Finding best accuracy in range 25-45
max_acc=0
best_k=0
scaler=StandardScaler()
x_train_scale=scaler.fit_transform(x_train)
x_test_scale=scaler.transform(x_test)
for i in range(25,45):
    knn=KNeighborsClassifier(n_neighbors=i)
    knn.fit(x_train_scale,y_train)
    y_pred=knn.predict(x_test_scale)
    acc=accuracy_score(y_test,y_pred)
    print(i,acc)
    if acc>max_acc:
        max_acc=acc
        best_k=i
print("Best K:",best_k,"with accuracy:",max_acc)

25 0.756578947368421
26 0.75
27 0.743421052631579
28 0.743421052631579
29 0.756578947368421
30 0.743421052631579
31 0.75
32 0.756578947368421
33 0.75
34 0.75
35 0.743421052631579
36 0.743421052631579
37 0.743421052631579
38 0.75
39 0.75
40 0.7302631578947368
41 0.75
42 0.743421052631579
43 0.743421052631579
44 0.7368421052631579
Best K: 25 with accuracy: 0.756578947368421


In [40]:
df=pd.read_csv("Fish.csv")

In [44]:
# Predict Species
x=df.drop(["Species"],axis=1)
y=df["Species"]
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
# w150,l112,l215,l314,h7,wi6
knn=KNeighborsClassifier(n_neighbors=5)
knn.fit(x_train,y_train)
y_pred=knn.predict(x_test)
acc=accuracy_score(y_test,y_pred)
y_pred_data=knn.predict([[150,12,15,14,7,6]])
print(y_pred_data)

['Perch']


c:\Users\yuvra\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
